# StockSense — Chronos-2 zero-shot benchmark

This notebook evaluates the pretrained `amazon/chronos-2` time-series foundation model as an **offline challenger** to StockSense's simple return baselines and existing XGBoost experiment. It does not change or deploy the web application's model.

The model receives only the 252 adjusted closing prices available at each forecast origin. It forecasts the next 10 trading-session positions. The terminal log-price forecast is converted to a 10-session return. Evaluation is chronological, and uncertainty calibration uses only outcomes before the evaluation year.

Start with `RUN_MODE = "pilot"`. A full daily walk-forward run is intentionally opt-in because it requires many thousands of foundation-model forecasts.

## Cell 1 — Install the Colab-only dependencies

In [ ]:
!pip -q install "chronos-forecasting>=2.3,<3" "pandas[pyarrow]>=2.2,<3" "scikit-learn>=1.5,<2"

print("Chronos and evaluation libraries installed.")

Chronos model weights are downloaded by Hugging Face only when the model-loading cell runs. They remain in the Colab runtime cache and are not added to the StockSense repository.

## Cell 2 — Imports and predeclared experiment settings

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from google.colab import files
from huggingface_hub import model_info
from sklearn.metrics import balanced_accuracy_score, mean_absolute_error, mean_squared_error, r2_score

MODEL_ID = "amazon/chronos-2"
CONTEXT_LENGTH = 252
HORIZON = 10
QUANTILES = [0.05, 0.5, 0.95]
NOMINAL_COVERAGE = 0.90
RUN_MODE = "pilot"  # Change to "full" only after the pilot succeeds.
PILOT_TICKERS = ["AAPL", "MSFT", "NVDA", "SPY", "AMZN", "JPM", "XOM", "JNJ"]
PILOT_ORIGIN_STRIDE = 21  # Approximately monthly origins.
FULL_ORIGIN_STRIDE = 1
FORECAST_BATCH_TASKS = 128
MODEL_BATCH_SIZE = 32
SEED = 42

assert RUN_MODE in {"pilot", "full"}
assert QUANTILES == sorted(QUANTILES) and QUANTILES[1] == 0.5
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
print({"run_mode": RUN_MODE, "device": DEVICE, "torch_dtype": str(DTYPE), "model": MODEL_ID})

The pilot uses eight diverse tickers and roughly monthly origins. It is a runtime and plumbing check, not evidence for model selection. The full mode evaluates every eligible ticker on every available origin.

## Cell 3 — Upload the immutable StockSense inputs

In [ ]:
print("Upload these two repository files:")
print("1. data/raw/historical_prices.parquet")
print("2. data/asset_universe.csv")
uploaded = files.upload()

price_candidates = [Path(name) for name in uploaded if Path(name).stem.startswith("historical_prices") and Path(name).suffix.lower() == ".parquet"]
universe_candidates = [Path(name) for name in uploaded if Path(name).stem.startswith("asset_universe") and Path(name).suffix.lower() == ".csv"]
assert len(price_candidates) == 1, "Upload exactly one historical_prices.parquet file."
assert len(universe_candidates) == 1, "Upload exactly one asset_universe.csv file."
PRICE_PATH = price_candidates[0]
UNIVERSE_PATH = universe_candidates[0]

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("Price SHA-256:", sha256(PRICE_PATH))
print("Universe SHA-256:", sha256(UNIVERSE_PATH))

## Cell 4 — Validate prices and establish the eligible universe

In [ ]:
raw_prices = pd.read_parquet(PRICE_PATH)
universe = pd.read_csv(UNIVERSE_PATH)

date_col = "Date" if "Date" in raw_prices.columns else "date"
ticker_col = "ticker"
price_col = "Adj Close" if "Adj Close" in raw_prices.columns else "adjusted_close"
required = {date_col, ticker_col, price_col}
assert required.issubset(raw_prices.columns), f"Missing price columns: {required - set(raw_prices.columns)}"

prices = raw_prices[[date_col, ticker_col, price_col]].rename(
    columns={date_col: "Date", ticker_col: "ticker", price_col: "adjusted_close"}
).copy()
prices["Date"] = pd.to_datetime(prices["Date"], errors="raise").dt.normalize()
prices["ticker"] = prices["ticker"].astype(str).str.upper().str.strip()
prices["adjusted_close"] = pd.to_numeric(prices["adjusted_close"], errors="raise")
prices = prices.sort_values(["ticker", "Date"]).reset_index(drop=True)

assert not prices.duplicated(["ticker", "Date"]).any(), "Duplicate ticker/date prices found."
assert np.isfinite(prices["adjusted_close"]).all(), "Non-finite adjusted prices found."
assert prices["adjusted_close"].gt(0).all(), "Adjusted prices must be positive for log transformation."

universe.columns = [str(column).strip() for column in universe.columns]
universe_ticker_col = next(column for column in universe.columns if column.lower() == "ticker")
eligibility_candidates = [column for column in universe.columns if column.lower() in {"forecast_eligible", "forecasting_eligible", "eligible_for_forecasting"}]
if eligibility_candidates:
    eligible_flag = universe[eligibility_candidates[0]].astype(str).str.lower().isin({"1", "true", "yes", "y"})
    eligible_tickers = set(universe.loc[eligible_flag, universe_ticker_col].astype(str).str.upper())
else:
    asset_type_candidates = [column for column in universe.columns if column.lower() == "asset_type"]
    eligible_tickers = set(universe[universe_ticker_col].astype(str).str.upper())
    if asset_type_candidates:
        non_vix = ~universe[asset_type_candidates[0]].astype(str).str.lower().isin({"index", "volatility"})
        eligible_tickers = set(universe.loc[non_vix, universe_ticker_col].astype(str).str.upper())

eligible_tickers &= set(prices["ticker"].unique())
assert eligible_tickers, "No eligible tickers matched the price file."

print("Price rows:", f"{len(prices):,}")
print("Eligible tickers:", len(eligible_tickers))
print("Date range:", prices.Date.min().date(), "through", prices.Date.max().date())
display(prices.head())

## Cell 5 — Build chronological calibration and evaluation tasks

In [ ]:
series = {}
for ticker, frame in prices.loc[prices.ticker.isin(eligible_tickers)].groupby("ticker", sort=True):
    frame = frame.sort_values("Date").reset_index(drop=True)
    series[ticker] = {
        "dates": frame.Date.to_numpy(dtype="datetime64[ns]"),
        "prices": frame.adjusted_close.to_numpy(dtype=float),
    }

def make_tasks(start: str, stop: str, *, stride: int) -> pd.DataFrame:
    start_ts, stop_ts = pd.Timestamp(start), pd.Timestamp(stop)
    rows = []
    selected = sorted(series) if RUN_MODE == "full" else [ticker for ticker in PILOT_TICKERS if ticker in series]
    for ticker in selected:
        dates = pd.DatetimeIndex(series[ticker]["dates"])
        candidate = np.flatnonzero((dates >= start_ts) & (dates < stop_ts))
        candidate = candidate[(candidate >= CONTEXT_LENGTH) & (candidate + HORIZON < len(dates))]
        for position in candidate[::stride]:
            label_end = dates[position + HORIZON]
            if label_end >= stop_ts:
                continue
            rows.append({
                "ticker": ticker,
                "origin_position": int(position),
                "Date": dates[position],
                "label_end_date": label_end,
            })
    return pd.DataFrame(rows).sort_values(["Date", "ticker"]).reset_index(drop=True)

stride = FULL_ORIGIN_STRIDE if RUN_MODE == "full" else PILOT_ORIGIN_STRIDE
fold_specs = {
    2024: {"calibration": ("2023-10-01", "2024-01-01"), "evaluation": ("2024-01-01", "2025-01-01")},
    2025: {"calibration": ("2024-10-01", "2025-01-01"), "evaluation": ("2025-01-01", "2026-01-01")},
    2026: {"calibration": ("2025-10-01", "2026-01-01"), "evaluation": ("2026-01-01", "2027-01-01")},
}

folds = {}
coverage_rows = []
for year, spec in fold_specs.items():
    calibration = make_tasks(*spec["calibration"], stride=stride)
    evaluation = make_tasks(*spec["evaluation"], stride=stride)
    folds[year] = {"calibration": calibration, "evaluation": evaluation}
    for partition, frame in folds[year].items():
        coverage_rows.append({"year": year, "partition": partition, "rows": len(frame), "tickers": frame.ticker.nunique(), "first_origin": frame.Date.min(), "last_origin": frame.Date.max(), "latest_label_end": frame.label_end_date.max()})

coverage = pd.DataFrame(coverage_rows)
display(coverage)
assert all((frame.label_end_date < pd.Timestamp(f"{year + 1}-01-01")).all() for year in folds for frame in [folds[year]["evaluation"]])
print("Chronological label-boundary checks: PASS")

The fourth-quarter calibration window precedes its evaluation year. Evaluation labels never choose a model or calibrate that year's uncertainty interval. Because Chronos remains frozen, there is no fitted training partition, but the ten-session label boundary is still enforced.

## Cell 6 — Download and load the frozen pretrained model

In [ ]:
from chronos import BaseChronosPipeline

MODEL_REVISION = model_info(MODEL_ID).sha
load_started = time.time()
pipeline = BaseChronosPipeline.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    device_map=DEVICE,
    torch_dtype=DTYPE,
)
print(f"Loaded {MODEL_ID}@{MODEL_REVISION} on {DEVICE} in {(time.time() - load_started):.1f} seconds.")

## Cell 7 — Define leakage-safe batched inference

In [ ]:
SYNTHETIC_DATES = pd.date_range("2000-01-01", periods=CONTEXT_LENGTH, freq="D")

def predict_task_batch(task_batch: pd.DataFrame) -> pd.DataFrame:
    contexts = []
    task_records = []
    for task_number, task in task_batch.reset_index(drop=True).iterrows():
        ticker = task.ticker
        position = int(task.origin_position)
        ticker_prices = series[ticker]["prices"]
        context_prices = ticker_prices[position - CONTEXT_LENGTH + 1 : position + 1]
        assert len(context_prices) == CONTEXT_LENGTH
        item_id = f"{ticker}|{pd.Timestamp(task.Date).date()}"
        contexts.append(pd.DataFrame({
            "item_id": item_id,
            "timestamp": SYNTHETIC_DATES,
            "log_price": np.log(context_prices),
        }))
        current = ticker_prices[position]
        future = ticker_prices[position + HORIZON]
        recent_log_returns = np.diff(np.log(context_prices[-21:]))
        scale_10d = max(float(np.std(recent_log_returns, ddof=1) * np.sqrt(HORIZON)), 0.005)
        historical_10d = ticker_prices[position - 9 : position + 1] / ticker_prices[position - 19 : position - 9] - 1
        task_records.append({
            "item_id": item_id, "ticker": ticker, "Date": pd.Timestamp(task.Date),
            "label_end_date": pd.Timestamp(task.label_end_date),
            "current_price": current, "actual_return_10d": future / current - 1,
            "zero_prediction": 0.0,
            "rolling_median_prediction": float(np.median(historical_10d)),
            "scale_10d": scale_10d,
        })

    context_df = pd.concat(contexts, ignore_index=True)
    forecast_df = pipeline.predict_df(
        context_df, prediction_length=HORIZON, quantile_levels=QUANTILES,
        id_column="item_id", timestamp_column="timestamp", target="log_price",
        batch_size=MODEL_BATCH_SIZE, context_length=CONTEXT_LENGTH, freq="D",
    )
    terminal = forecast_df.sort_values(["item_id", "timestamp"]).groupby("item_id", as_index=False).tail(1)
    terminal = terminal.set_index("item_id")
    output = pd.DataFrame(task_records).set_index("item_id")
    output["chronos_prediction"] = np.exp(terminal["predictions"] - np.log(output["current_price"])) - 1
    output["native_lower"] = np.exp(terminal[str(QUANTILES[0])] - np.log(output["current_price"])) - 1
    output["native_upper"] = np.exp(terminal[str(QUANTILES[-1])] - np.log(output["current_price"])) - 1
    return output.reset_index(drop=True)

def forecast_tasks(tasks: pd.DataFrame) -> pd.DataFrame:
    completed = []
    for start in range(0, len(tasks), FORECAST_BATCH_TASKS):
        stop = min(start + FORECAST_BATCH_TASKS, len(tasks))
        completed.append(predict_task_batch(tasks.iloc[start:stop]))
        print(f"Forecasted {stop:,}/{len(tasks):,} tasks", flush=True)
    return pd.concat(completed, ignore_index=True) if completed else pd.DataFrame()

print("Inference functions defined; no forecasts computed yet.")

Trading sessions are mapped to regular synthetic sequence positions. This prevents weekends and market holidays from being mistaken for missing observations. Only ordering and observed values enter the zero-shot model.

## Cell 8 — One-task smoke test

In [ ]:
smoke_source = next(folds[year]["evaluation"] for year in sorted(folds) if len(folds[year]["evaluation"]))
smoke = forecast_tasks(smoke_source.head(1))
display(smoke)
assert np.isfinite(smoke[["chronos_prediction", "native_lower", "native_upper"]].to_numpy()).all()
assert (smoke.native_lower <= smoke.native_upper).all()
print("Chronos smoke test: PASS")

## Cell 9 — Run the chronological pilot or full benchmark

In [ ]:
predictions = {}
for year in sorted(folds):
    print(f"\n=== {year}: calibration ===")
    calibration = forecast_tasks(folds[year]["calibration"])
    print(f"=== {year}: evaluation ===")
    evaluation = forecast_tasks(folds[year]["evaluation"])

    calibration_scores = np.abs(calibration.actual_return_10d - calibration.chronos_prediction) / calibration.scale_10d
    rank = min(len(calibration_scores), math.ceil((len(calibration_scores) + 1) * NOMINAL_COVERAGE))
    conformal_q = float(np.sort(calibration_scores)[rank - 1])
    evaluation["calibrated_lower"] = evaluation.chronos_prediction - conformal_q * evaluation.scale_10d
    evaluation["calibrated_upper"] = evaluation.chronos_prediction + conformal_q * evaluation.scale_10d
    evaluation["year"] = year
    evaluation["calibration_score_quantile"] = conformal_q
    predictions[year] = evaluation

all_predictions = pd.concat(predictions.values(), ignore_index=True)
print("Completed rows:", f"{len(all_predictions):,}")
display(all_predictions.head())

Two intervals are retained: the model's native 5th–95th percentile interval and a StockSense-style volatility-scaled interval calibrated entirely on the preceding fourth quarter. The native interval is evaluated rather than assumed to be calibrated.

## Cell 10 — Compare return accuracy, direction, and uncertainty

In [ ]:
def interval_score(y, lower, upper, alpha=0.10):
    width = upper - lower
    return width + (2 / alpha) * (lower - y) * (y < lower) + (2 / alpha) * (y - upper) * (y > upper)

def score_predictions(frame: pd.DataFrame, prediction_column: str, model_name: str) -> dict:
    y = frame.actual_return_10d.to_numpy()
    p = frame[prediction_column].to_numpy()
    actual_up, predicted_up = y > 0, p > 0
    return {
        "Model": model_name, "Rows": len(frame),
        "MAE (pp)": mean_absolute_error(y, p) * 100,
        "RMSE (pp)": mean_squared_error(y, p) ** 0.5 * 100,
        "R2": r2_score(y, p),
        "Direction accuracy (%)": np.mean(actual_up == predicted_up) * 100,
        "Balanced accuracy (%)": balanced_accuracy_score(actual_up, predicted_up) * 100,
        "Mean prediction (%)": np.mean(p) * 100,
    }

metric_rows = []
interval_rows = []
for year, frame in predictions.items():
    for column, name in (("zero_prediction", "Zero return"), ("rolling_median_prediction", "Rolling median"), ("chronos_prediction", "Chronos-2")):
        metric_rows.append({"Year": year, **score_predictions(frame, column, name)})
    y = frame.actual_return_10d.to_numpy()
    for lower_col, upper_col, name in (("native_lower", "native_upper", "Chronos native 90%"), ("calibrated_lower", "calibrated_upper", "Chronos calibrated 90%")):
        lower, upper = frame[lower_col].to_numpy(), frame[upper_col].to_numpy()
        interval_rows.append({
            "Year": year, "Interval": name, "Rows": len(frame),
            "Coverage (%)": np.mean((y >= lower) & (y <= upper)) * 100,
            "Mean width (pp)": np.mean(upper - lower) * 100,
            "Mean interval score (pp)": np.mean(interval_score(y, lower, upper)) * 100,
        })

metrics = pd.DataFrame(metric_rows)
interval_metrics = pd.DataFrame(interval_rows)
print("=== POINT FORECASTS ===")
display(metrics.round(4))
print("=== UNCERTAINTY ===")
display(interval_metrics.round(4))

## Cell 11 — Date-block bootstrap against the rolling-median baseline

In [ ]:
def date_block_bootstrap(frame: pd.DataFrame, block_length: int, repeats: int = 1000, seed: int = SEED):
    losses = frame.assign(
        gain=(np.abs(frame.actual_return_10d - frame.rolling_median_prediction) - np.abs(frame.actual_return_10d - frame.chronos_prediction)) * 100
    ).groupby("Date").gain.agg(["sum", "count"])
    if len(losses) < block_length:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    draws = []
    block_count = math.ceil(len(losses) / block_length)
    for _ in range(repeats):
        starts = rng.integers(0, len(losses) - block_length + 1, size=block_count)
        indices = np.concatenate([np.arange(start, start + block_length) for start in starts])[:len(losses)]
        selected = losses.iloc[indices]
        draws.append(selected["sum"].sum() / selected["count"].sum())
    return float(frame.assign(gain=(np.abs(frame.actual_return_10d - frame.rolling_median_prediction) - np.abs(frame.actual_return_10d - frame.chronos_prediction)) * 100).gain.mean()), *np.quantile(draws, [0.025, 0.975])

bootstrap_rows = []
for year, frame in predictions.items():
    for block in (10, 20, 40):
        gain, lower, upper = date_block_bootstrap(frame, block)
        bootstrap_rows.append({"Year": year, "Block sessions": block, "MAE gain vs rolling median (pp)": gain, "95% lower": lower, "95% upper": upper, "Excludes zero": bool(lower > 0 or upper < 0) if np.isfinite(lower) else False})
bootstrap_results = pd.DataFrame(bootstrap_rows)
display(bootstrap_results.round(4))
print("Positive gain favors Chronos-2. A confidence interval crossing zero does not establish improvement.")

## Cell 12 — Visualize model errors and prediction variation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for year, frame in predictions.items():
    axes[0].scatter(frame.actual_return_10d * 100, frame.chronos_prediction * 100, s=10, alpha=0.35, label=str(year))
axes[0].axline((0, 0), slope=1, color="black", linestyle="--", linewidth=1)
axes[0].set(xlabel="Actual 10-session return (%)", ylabel="Chronos-2 prediction (%)", title="Predicted versus actual")
axes[0].legend()
metrics.pivot(index="Year", columns="Model", values="MAE (pp)").plot(kind="bar", ax=axes[1])
axes[1].set(title="MAE by evaluation year", ylabel="MAE (percentage points)")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## Cell 13 — Export an auditable experimental record

In [ ]:
OUTPUT_DIR = Path("/content/stocksense_chronos2_benchmark")
OUTPUT_DIR.mkdir(exist_ok=True)
all_predictions.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
interval_metrics.to_csv(OUTPUT_DIR / "interval_metrics.csv", index=False)
bootstrap_results.to_csv(OUTPUT_DIR / "bootstrap_results.csv", index=False)

record = {
    "experiment": "StockSense Chronos-2 zero-shot benchmark",
    "run_mode": RUN_MODE, "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "context_length": CONTEXT_LENGTH, "horizon": HORIZON,
    "target_representation": "log adjusted close; terminal forecast transformed to return",
    "quantiles": QUANTILES, "nominal_coverage": NOMINAL_COVERAGE,
    "price_sha256": sha256(PRICE_PATH), "universe_sha256": sha256(UNIVERSE_PATH),
    "device": DEVICE, "python": platform.python_version(),
    "torch": torch.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
    "limitations": [
        "Retrospective zero-shot evaluation; public pretraining overlap cannot be ruled out",
        "Pilot mode is sparse and cannot select a model",
        "Adjusted prices may include later vendor revisions",
        "Interval calibration is empirical and does not guarantee future coverage",
        "No result in this notebook is financial advice",
    ],
}
(OUTPUT_DIR / "experiment.json").write_text(json.dumps(record, indent=2))
!cd /content && zip -qr stocksense_chronos2_benchmark.zip stocksense_chronos2_benchmark
print("Saved:", OUTPUT_DIR)
files.download("/content/stocksense_chronos2_benchmark.zip")

## Decision rule

Do not replace the deployed XGBoost package from the pilot. Run the full experiment and promote Chronos only if it repeatedly lowers MAE/RMSE relative to the rolling-median baseline, its date-block confidence intervals support the improvement, balanced direction accuracy exceeds chance, and calibrated interval coverage is acceptably close to 90%. A large model name or a single direction-accuracy percentage is not sufficient evidence.

If Chronos passes, the safest first deployment remains offline batch generation of the existing forecast CSV/JSON contract. Keep PyTorch and downloaded model weights out of the FastAPI request process.